In [1]:
from google.colab import files
import pandas as pd


In [3]:


beneficiary = pd.read_csv("/content/DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv")
inpatient   = pd.read_csv("/content/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv")
outpatient  = pd.read_csv("/content/DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv")

print("Beneficiary shape:", beneficiary.shape)
print("Inpatient shape:",   inpatient.shape)
print("Outpatient shape:",  outpatient.shape)

/tmp/ipykernel_5909/3034430063.py:3: DtypeWarning: Columns (21,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  outpatient  = pd.read_csv("/content/DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv")


Beneficiary shape: (116352, 32)
Inpatient shape: (66773, 81)
Outpatient shape: (790790, 76)


In [4]:
# Join inpatient with beneficiary on DESYNPUF_ID
inpatient_merged = inpatient.merge(beneficiary, on='DESYNPUF_ID', how='left')

# Join outpatient with beneficiary
outpatient_merged = outpatient.merge(beneficiary, on='DESYNPUF_ID', how='left')

# Add a claim type column
inpatient_merged['CLAIM_TYPE']  = 'Inpatient'
outpatient_merged['CLAIM_TYPE'] = 'Outpatient'

# Find common columns to combine both
common_cols = list(set(inpatient_merged.columns) & set(outpatient_merged.columns))
combined = pd.concat([inpatient_merged[common_cols],
                      outpatient_merged[common_cols]], ignore_index=True)

print("Combined shape:", combined.shape)
print(combined.head())

Combined shape: (857563, 106)
   BENE_SMI_CVRAGE_TOT_MONS HCPCS_CD_35 HCPCS_CD_2 HCPCS_CD_4 HCPCS_CD_21  \
0                        12         NaN        NaN        NaN         NaN   
1                        12         NaN        NaN        NaN         NaN   
2                        12         NaN        NaN        NaN         NaN   
3                        12         NaN        NaN        NaN         NaN   
4                        12         NaN        NaN        NaN         NaN   

   SP_DEPRESSN HCPCS_CD_28 HCPCS_CD_1  SP_DIABETES  OP_PHYSN_NPI  ...  \
0            2         NaN        NaN            2           NaN  ...   
1            2         NaN        NaN            2           NaN  ...   
2            2         NaN        NaN            2  6.119985e+08  ...   
3            2         NaN        NaN            2           NaN  ...   
4            2         NaN        NaN            2  1.960860e+09  ...   

  HCPCS_CD_20 CLM_FROM_DT       DESYNPUF_ID SP_CNCR HCPCS_CD_30 HCPC

In [5]:
# Rename key columns for readability
combined.rename(columns={
    'CLM_PMT_AMT'      : 'CLAIM_PAYMENT_AMT',
    'CLM_FROM_DT'      : 'CLAIM_START_DATE',
    'CLM_THRU_DT'      : 'CLAIM_END_DATE',
    'ADMTNG_ICD9_DGNS_CD' : 'PRIMARY_DIAGNOSIS_CODE'
}, inplace=True, errors='ignore')

# Convert dates
combined['CLAIM_START_DATE'] = pd.to_datetime(combined['CLAIM_START_DATE'], format='%Y%m%d', errors='coerce')
combined['CLAIM_END_DATE']   = pd.to_datetime(combined['CLAIM_END_DATE'],   format='%Y%m%d', errors='coerce')

# Calculate processing time in days
combined['PROCESSING_DAYS'] = (combined['CLAIM_END_DATE'] - combined['CLAIM_START_DATE']).dt.days

# Flag potential denials (zero payment = likely denied)
combined['DENIAL_FLAG'] = combined['CLAIM_PAYMENT_AMT'].apply(lambda x: 1 if x == 0 else 0)

print(combined[['CLAIM_TYPE','CLAIM_PAYMENT_AMT','PROCESSING_DAYS','DENIAL_FLAG']].describe())

       CLAIM_PAYMENT_AMT  PROCESSING_DAYS    DENIAL_FLAG
count      857563.000000    846242.000000  857563.000000
mean         1007.255315         1.723624       0.037624
std          3640.546992         4.879575       0.190285
min         -8000.000000         0.000000       0.000000
25%            40.000000         0.000000       0.000000
50%            80.000000         0.000000       0.000000
75%           300.000000         0.000000       0.000000
max         57000.000000        35.000000       1.000000


In [6]:
# Remove negative payments (these are adjustments, not real claims)
combined = combined[combined['CLAIM_PAYMENT_AMT'] >= 0]

# Remove nulls in key columns
combined = combined.dropna(subset=['PRIMARY_DIAGNOSIS_CODE', 'CLAIM_START_DATE'])

# Add year column for trend analysis in Tableau
combined['CLAIM_YEAR'] = combined['CLAIM_START_DATE'].dt.year

# Add cost buckets for easier Tableau filtering
combined['COST_BUCKET'] = pd.cut(combined['CLAIM_PAYMENT_AMT'],
                                  bins=[0, 100, 500, 1000, 5000, 60000],
                                  labels=['<$100', '$100-500', '$500-1K', '$1K-5K', '$5K+'])

print("Clean shape:", combined.shape)
print("\nClaims by year:")
print(combined['CLAIM_YEAR'].value_counts().sort_index())
print("\nDenial rate by claim type:")
print(combined.groupby('CLAIM_TYPE')['DENIAL_FLAG'].mean().round(4))

Clean shape: (260063, 110)

Claims by year:
CLAIM_YEAR
2007       307
2008    144320
2009     90568
2010     24868
Name: count, dtype: int64

Denial rate by claim type:
CLAIM_TYPE
Inpatient     0.0321
Outpatient    0.0407
Name: DENIAL_FLAG, dtype: float64


In [8]:
# Remove 2007 spillover records
combined = combined[combined['CLAIM_YEAR'] >= 2008]

# Final export
combined.to_csv('healthcare_claims_tableau_ready.csv', index=False)

In [9]:
# Check denial rate per diagnosis code
denial_by_code = combined.groupby('PRIMARY_DIAGNOSIS_CODE').agg(
    total_claims = ('DENIAL_FLAG', 'count'),
    denied_claims = ('DENIAL_FLAG', 'sum')
).reset_index()

denial_by_code['denial_rate'] = denial_by_code['denied_claims'] / denial_by_code['total_claims']

# Show codes with MIXED denial rates (not 0 or 1)
mixed = denial_by_code[(denial_by_code['denial_rate'] > 0) &
                        (denial_by_code['denial_rate'] < 1) &
                        (denial_by_code['total_claims'] >= 100)]

print(mixed.sort_values('denial_rate', ascending=False).head(20))

     PRIMARY_DIAGNOSIS_CODE  total_claims  denied_claims  denial_rate
2204                   5259           166             63     0.379518
4710                   V681           103             23     0.223301
4716                   V700           276             56     0.202899
4564                  V4501           127             19     0.149606
4792                  V8281           240             34     0.141667
4773                  V7651          1036            133     0.128378
4743                   V726           264             33     0.125000
3510                  78830           107             13     0.121495
4765                   V762           239             29     0.121339
4769                  V7644           216             26     0.120370
3517                  78841           242             28     0.115702
3581                   7921           136             14     0.102941
4626                  V5489           148             15     0.101351
975                 

In [10]:
# Filter to only meaningful codes - no V codes, minimum 100 claims, mixed denial rates
denial_by_code = combined.groupby('PRIMARY_DIAGNOSIS_CODE').agg(
    total_claims=('DENIAL_FLAG', 'count'),
    denied_claims=('DENIAL_FLAG', 'sum')
).reset_index()

denial_by_code['denial_rate'] = denial_by_code['denied_claims'] / denial_by_code['total_claims']

# Keep only:
# 1. Numeric codes only (no V or E codes)
# 2. At least 100 claims
# 3. Denial rate between 1% and 99% (meaningful rates)
good_codes = denial_by_code[
    (~denial_by_code['PRIMARY_DIAGNOSIS_CODE'].str.startswith(('V', 'E'), na=False)) &
    (denial_by_code['total_claims'] >= 100) &
    (denial_by_code['denial_rate'] > 0.01) &
    (denial_by_code['denial_rate'] < 0.99)
]

print(f"Valid codes found: {len(good_codes)}")
print("\nTop 15 by denial rate:")
print(good_codes.sort_values('denial_rate', ascending=False).head(15))

# Filter main dataset to only these good codes
combined_clean = combined[combined['PRIMARY_DIAGNOSIS_CODE'].isin(good_codes['PRIMARY_DIAGNOSIS_CODE'])]
print("\nClean dataset shape:", combined_clean.shape)

Valid codes found: 349

Top 15 by denial rate:
     PRIMARY_DIAGNOSIS_CODE  total_claims  denied_claims  denial_rate
2204                   5259           166             63     0.379518
3510                  78830           107             13     0.121495
3517                  78841           242             28     0.115702
3581                   7921           136             14     0.102941
975                   29570           602             60     0.099668
2274                  53550           101             10     0.099010
820                   27801           133             13     0.097744
105                    1101           126             12     0.095238
1602                  37991           117             11     0.094017
3017                  71940           127             11     0.086614
2028                   4619           105              9     0.085714
2603                  61179           117             10     0.085470
4009                   8830           200  

In [13]:
# Fix the SettingWithCopyWarning by making a proper copy
combined_clean = combined_clean.copy()

# Expanded ICD-9 lookup with more codes from your dataset
icd9_lookup = {
    '5259':  'Tonsil/Adenoid Disorders',
    '78830': 'Fecal Incontinence',
    '78841': 'Fecal Urgency',
    '7921':  'Elevated Blood Glucose',
    '29570': 'Drug-Induced Mood Disorder',
    '27801': 'Overweight',
    '1101':  'Nail Fungal Infection',
    '37991': 'Eye Disorder NOS',
    '71940': 'Unspecified Joint Disorder',
    '4619':  'Acute Sinusitis',
    '61179': 'Prostate Disorder',
    '8830':  'Open Wound - Finger',
    '4260':  'Atrioventricular Block',
    '6272':  'Excessive Menstruation',
    '53550': 'Intestinal Disorder NOS',
    # Additional codes from your dataset
    '7866':  'Hiccough/Cough Symptoms',
    '29590': 'Unspecified Psychosis',
    '5849':  'Acute Kidney Failure',
    '78079': 'Malaise & Fatigue',
    '78097': 'Other General Symptoms',
    '49392': 'Asthma - Unspecified',
    '4019':  'Hypertension - Unspecified',
    '72989': 'Unspecified Back Disorder',
    '42731': 'Atrial Fibrillation',
    '5693':  'Rectal/Anal Fistula',
    '49121': 'Obstructive Chronic Bronchitis',
    '51881': 'Acute Respiratory Failure',
    '7802':  'Syncope & Collapse',
    '41519': 'Pulmonary Embolism',
    '78009': 'Altered Consciousness NOS',
    '25080': 'Diabetes with Complications',
    '4280':  'Congestive Heart Failure',
    '7140':  'Rheumatoid Arthritis'
}

# Apply mapping safely
combined_clean['DIAGNOSIS_DESCRIPTION'] = combined_clean['PRIMARY_DIAGNOSIS_CODE'].map(icd9_lookup)
combined_clean['DIAGNOSIS_DESCRIPTION'] = combined_clean['DIAGNOSIS_DESCRIPTION'].fillna(
    'Code: ' + combined_clean['PRIMARY_DIAGNOSIS_CODE']
)

# Verify
print(combined_clean[['PRIMARY_DIAGNOSIS_CODE', 'DIAGNOSIS_DESCRIPTION']].drop_duplicates().head(20))
print("\nMapped descriptions:", combined_clean['DIAGNOSIS_DESCRIPTION'].str.startswith('Code:').value_counts())

   PRIMARY_DIAGNOSIS_CODE           DIAGNOSIS_DESCRIPTION
1                    7866         Hiccough/Cough Symptoms
3                   29590           Unspecified Psychosis
4                    5849            Acute Kidney Failure
5                   78079               Malaise & Fatigue
6                   78097          Other General Symptoms
7                   49392            Asthma - Unspecified
10                   4019      Hypertension - Unspecified
11                   4260          Atrioventricular Block
13                  72989       Unspecified Back Disorder
14                  42731             Atrial Fibrillation
15                   5693             Rectal/Anal Fistula
16                  49121  Obstructive Chronic Bronchitis
17                  51881       Acute Respiratory Failure
20                   7802              Syncope & Collapse
21                  29570      Drug-Induced Mood Disorder
24                  41519              Pulmonary Embolism
25            

In [16]:
# Add readable denial status column
combined_clean = combined_clean.copy()
combined_clean['DENIAL_STATUS'] = combined_clean['DENIAL_FLAG'].map({
    0: 'Approved',
    1: 'Denied'
})

combined_clean.to_csv('healthcare_claims_v5.csv', index=False)
files.download('healthcare_claims_v5.csv')
print("✅ Done!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done!
